# How to find material weakness disclosures in 10-Qs and 10-Ks

Search for companies that reported a material weakness in internal control and concluded their disclosure controls were not effective, and tell new problems from old ones.

Read it with charts at https://www.quantisnow.com/sec-api/guides/material-weakness-disclosures. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

Every 10-Q and 10-K ends with a controls section. When management finds a material weakness, a gap that could let a material error into the financial statements, it has to say so, and it usually concludes that disclosure controls were not effective. Weaknesses often come before restatements and auditor changes, so analysts and short sellers watch for new ones.

## The query

`"material weakness"` alone matches every filing that describes one remediated years ago. Adding `"not effective"` keeps the filings where the problem is current.

### Who reported a material weakness and said their controls were not effective?

A material weakness in internal control often comes before a restatement or an auditor change. Requiring "not effective" drops the filings that only mention a weakness from years ago.

```shell
curl -X POST https://www.quantisnow.com/api/v1/full-text \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "\"material weakness\" AND \"not effective\"",
  "formTypes": ["10-Q", "10-Q/A", "10-K", "10-K/A"],
  "sort": "filedAt:desc",
  "size": 5,
  "facets": true
}'
```

Response (real, first 2 of 240):

```json
{
  "total": {
    "value": 240,
    "relation": "eq"
  },
  "took": 257,
  "data": [
    {
      "id": "0001477932-26-007251",
      "kind": "filing",
      "source": "SEC",
      "accessionNo": "0001477932-26-007251",
      "formType": "10-Q",
      "filedAt": "2026-11-23T17:27:21-05:00",
      "cik": "0001425627",
      "companyName": "SOBR Safe Inc.",
      "ticker": "SOBR",
      "tickers": [
        "SOBR"
      ],
      "items": [],
      "description": "SEC Form 10-Q filed by SOBR Safe Inc.",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1425627/000147793226007251/0001477932-26-007251-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/sec-form-10-q-filed-by-sobr-safe-inc-6804398",
      "score": null,
      "highlights": [
        "The identified <em>material weakness</em> is primarily driven by an insufficient number of personnel to conduct an effective system of internal controls leading to a lack of segregation of duties where a few or one individual may",
        "The <em>material weakness</em> in internal controls over financial reporting may result in the possibility that a material misstatement of the Company's interim financial statements at September 30, 2026, and future reporting periods"
      ]
    },
    {
      "id": "0001104659-26-131184",
      "kind": "filing",
      "source": "SEC",
      "accessionNo": "0001104659-26-131184",
      "formType": "10-Q",
      "filedAt": "2026-11-23T17:22:34-05:00",
      "cik": "0001479419",
      "companyName": "KALA BIO Inc.",
      "ticker": "KALA",
      "tickers": [
        "KALA"
      ],
      "items": [],
      "description": "SEC Form 10-Q filed by KALA BIO Inc.",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1479419/000110465926131184/0001104659-26-131184-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/sec-form-10-q-filed-by-kala-bio-inc-6804382",
      "score": null,
      "highlights": [
        "<em>effective</em> at the reasonable assurance level.",
        "As described in our Annual Report on Form 10-K for the year ended December 31, 2025, we previously identified a <em>material weakness</em> in our internal control over financial reporting that insufficient personnel following the",
        "This <em>material weakness</em> remained un-remediated as of September 30, 2026. We have initiated remediation activities and will continue to evaluate and enhance our controls."
      ]
    }
  ],
  "facets": {
    "formType": [
      {
        "value": "10-Q",
        "count": 216
      },
      {
        "value": "10-K",
        "count": 14
      },
      {
        "value": "10-Q/A",
        "count": 6
      },
      {
        "value": "10-K/A",
        "count": 4
      }
    ],
    "month": [
      {
        "value": "2026-11",
        "count": 233
      },
      {
        "value": "2026-10",
        "count": 7
      }
    ],
    "companyName": [
      {
        "value": "Solesence Inc.",
        "count": 6
      },
      {
        "value": "Optimum Communications Inc.",
        "count": 3
      },
      {
        "value": "Brand Engagement Network Inc.",
        "count": 2
      },
      {
        "value": "AAON Inc.",
        "count": 1
      },
      {
        "value": "ABVC BioPharma Inc.",
        "count": 1
      },
      {
        "value": "AEVEX Corp.",
        "count": 1
      },
      {
        "value": "AIRO Group Holdings Inc.",
        "count": 1
      },
      {
        "value": "Abundia Global Impact Group Inc.",
        "count": 1
      },
      {
        "value": "Aebi Schmidt Holding AG",
        "count": 1
      },
      {
        "value": "Aeluma Inc.",
        "count": 1
      }
    ],
    "source": [
      {
        "value": "SEC",
        "count": 240
      }
    ]
  }
}
```

## New or ongoing?

A weakness reported last quarter shows up again until it is fixed. To find the new ones, compare against the company's previous filing: run the same query with `ticker:XYZ` and a `filedAt` range covering the quarter before, and keep companies with no earlier match. A 10-Q/A or 10-K/A in the results is worth reading first: amended controls sections often follow a restatement.

In [ ]:
# URL and AUTH as in the Python guide: /sec-api/guides/search-sec-filings-python
def mentions(ticker, start, end):
    page = requests.post(URL, headers=AUTH, json={
        "query": f'"material weakness" AND "not effective" AND ticker:{ticker} AND filedAt:[{start} TO {end}]',
        "size": 1,
    }).json()
    return page["total"]["value"] > 0

new = [t for t in tickers if not mentions(t, "2026-06-01", "2026-09-30")]

## What tends to come next

A restatement is announced in an 8-K under item 4.02 (non-reliance on previous financial statements), and an auditor change under 4.01. The [8-K events endpoint](https://www.quantisnow.com/sec-api/8-k-events) filters on both, so you can check which companies on your weakness list later filed one.